In [10]:

import torch
import nltk
import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sklearn.metrics import f1_score, accuracy_score

nltk.download("punkt_tab")

if torch.backends.mps.is_available():
  device = torch.device('mps')

  print(device)

else:
  print("No GPU available, using the CPU instead")
  device = torch.device("cpu")

  

mps


[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/fepriyadi/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## 1) Data Extraction and Sampling
Extract raw data, refresh destination folders, and build train/dev/test samples.

In [6]:
import os
import random
import shutil
from pathlib import Path

# Configuration
TOTAL_FILES_TO_SAMPLE = 20000
train_ratio = 0.70
dev_ratio = 0.10
test_ratio = 0.20

# Paths
local_base_path = 'dataset/canonical'
sampled_path = './dataset/sampled_20k'

if os.path.exists(sampled_path):
    print(f"Directory already exists: {sampled_path}. Skipping sampling/copying.")
else:
    # Create the directory structure
    os.makedirs(sampled_path, exist_ok=True)
    for split in ['train', 'dev', 'test']:
        os.makedirs(os.path.join(sampled_path, split), exist_ok=True)

    # Calculate counts
    train_count = int(TOTAL_FILES_TO_SAMPLE * train_ratio)
    dev_count = int(TOTAL_FILES_TO_SAMPLE * dev_ratio)
    test_count = TOTAL_FILES_TO_SAMPLE - train_count - dev_count  # Ensure it adds up exactly

    splits = {
        'train': train_count,
        'dev': dev_count,
        'test': test_count
    }

    print(f"Target distribution -> Train: {train_count}, Dev: {dev_count}, Test: {test_count}\n")

    for split, count in splits.items():
        src_dir = os.path.join(local_base_path, split)
        dst_dir = os.path.join(sampled_path, split)

        if os.path.exists(src_dir):
            # Get all json files in the source directory
            all_files = [f for f in os.listdir(src_dir) if f.endswith('.json')]
            print(f"Found {len(all_files)} files in local {split} folder.")

            # Sample files (or take all if count > available files)
            files_to_copy = random.sample(all_files, min(count, len(all_files)))

            print(f"Copying {len(files_to_copy)} files to Drive {split} folder...")
            for f in files_to_copy:
                shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
            print(f"Finished copying {split}.\n")
        else:
            print(f"Warning: Source directory {src_dir} not found!")

    print("Sampling and copying process completed!")

Directory already exists: ./dataset/sampled_20k. Skipping sampling/copying.


## 2) Data Sanity Check
Inspect one sample file to verify schema and content shape.

In [7]:
import os
import json

sample_dir = 'dataset/sampled_20k/train'

if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.json')]
    if sample_files:
        sample_file_path = os.path.join(sample_dir, sample_files[0])
        print(f"Reading sample file: {sample_file_path}\n")

        with open(sample_file_path, 'r') as f:
            data = json.load(f)

        print("JSON Keys:", list(data.keys()))
        print("\nFull JSON Content:")
        print(json.dumps(data, indent=2))
    else:
        print("No JSON files found in the directory.")
else:
    print(f"Directory not found: {sample_dir}")

Reading sample file: dataset/sampled_20k/train/176337.json

JSON Keys: ['id', 'url', 'clean_article', 'clean_summary', 'extractive_summary']

Full JSON Content:
{
  "id": 176337,
  "url": "https://www.liputan6.com/news/read/176337/pengidap-gangguan-jiwa-di-garut-bertambah",
  "clean_article": [
    [
      "Liputan6",
      ".",
      "com",
      ",",
      "Garut",
      ":",
      "Jumlah",
      "orang",
      "yang",
      "mengidap",
      "gangguan",
      "jiwa",
      "di",
      "Kecamatan",
      "Kersamanah",
      ",",
      "Garut",
      ",",
      "Jawa",
      "Barat",
      ",",
      "terus",
      "bertambah",
      "."
    ],
    [
      "Berdasarkan",
      "hasil",
      "pemantauan",
      "aparat",
      "kecamatan",
      ",",
      "setidaknya",
      "di",
      "kecamatan",
      "ini",
      "ada",
      "sekitar",
      "80",
      "orang",
      "yang",
      "mengidap",
      "gangguan",
      "jiwa",
      "."
    ],
    [
      "Mereka",
      "terseb

## 3) Load and Structure Dataset
Load JSON files into pandas DataFrames for each split.

In [8]:
import os
import json
import pandas as pd
import re
from tqdm.notebook import tqdm

def remove_news_header_robust(text):
    return re.sub(
        r"^Liputan6\s*\.\s*com\s*,\s*[^:]+:\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

def load_data_split(split_path):
    records = []
    if not os.path.exists(split_path):
        print(f"Path not found: {split_path}")
        return pd.DataFrame()

    files = [f for f in os.listdir(split_path) if f.endswith('.json')]
    for f in tqdm(files, desc=f"Loading {os.path.basename(split_path)}"):
        with open(os.path.join(split_path, f), 'r') as file:
            data = json.load(file)

            # Form the full text from nested lists
            article = " ".join([" ".join(sentence) for sentence in data.get('clean_article', [])])
            summary = " ".join([" ".join(sentence) for sentence in data.get('clean_summary', [])])

            # Apply preprocessing to remove news headers
            article = remove_news_header_robust(article)

            records.append({
                'id': data.get('id'),
                'url': data.get('url'),
                'article': article,
                'summary': summary,
                'extractive_summary': data.get('extractive_summary')
            })

    return pd.DataFrame(records)

# Base directory where your sampled data is stored
base_dir = 'dataset/sampled_20k'

# Load the datasets
print("Loading datasets into Pandas DataFrames...")
df_train = load_data_split(os.path.join(base_dir, 'train'))
df_dev = load_data_split(os.path.join(base_dir, 'dev'))
df_test = load_data_split(os.path.join(base_dir, 'test'))

print("\nData Loading Complete!")
print(f"Train size: {len(df_train)}")
print(f"Dev size: {len(df_dev)}")
print(f"Test size: {len(df_test)}")

# Preview the train dataframe
display(df_train.head())

Loading datasets into Pandas DataFrames...


Loading train:   0%|          | 0/14000 [00:00<?, ?it/s]

Loading dev:   0%|          | 0/2000 [00:00<?, ?it/s]

Loading test:   0%|          | 0/4000 [00:00<?, ?it/s]


Data Loading Complete!
Train size: 14000
Dev size: 2000
Test size: 4000


,id,url,article,summary,extractive_summary
0,176337,https://www.liputan6.com/news/read/176337/peng...,Jumlah orang yang mengidap gangguan jiwa di Ke...,Setidaknya ada sekitar 80 orang yang mengidap ...,"[1, 2]"
1,288128,https://www.liputan6.com/news/read/288128/dua-...,Dua dari sepuluh korban kebocoran tabung gas d...,Dua dari sepuluh korban kebocoran tabung gas d...,"[0, 1]"
2,238831,https://www.liputan6.com/news/read/238831/wapr...,"Wakil Presiden Jusuf Kalla optimistis , pasoka...","Wakil Presiden Jusuf Kalla optimistis , pasoka...","[0, 2]"
3,41942,https://www.liputan6.com/news/read/41942/warga...,Warga Nanggroe Aceh Darussalam meminta TNI tak...,Elemen masyarakat Aceh menginginkan pemerintah...,"[1, 21]"
4,124847,https://www.liputan6.com/news/read/124847/pera...,Helm atau pelindung kepala biasanya terbuat da...,"Seorang perajin di Tuban , Jatim , berhasil me...","[1, 2]"


## Exploratory Data Analysis
Understanding the corpus before modeling

In [25]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from collections import Counter

# Combine all splits
df_all = pd.concat([df_train, df_dev, df_test], ignore_index=True)

# Tokenize using simple whitespace split for token counts
df_all['doc_tokens'] = df_all['article'].apply(lambda x: len(x.split()))
df_all['sum_tokens'] = df_all['summary'].apply(lambda x: len(x.split()))

total_docs = len(df_all)
avg_doc_tokens = int(round(df_all['doc_tokens'].mean()))
avg_sum_tokens = int(round(df_all['sum_tokens'].mean()))
compression_ratio = round(avg_doc_tokens / avg_sum_tokens, 1)

# Document length distribution bins
bins = [0, 200, 400, 600, 800, float('inf')]
bin_labels = ['< 200 tokens', '200-400 tokens', '400-600 tokens', '600-800 tokens', '> 800 tokens']
df_all['doc_bin'] = pd.cut(df_all['doc_tokens'], bins=bins, labels=bin_labels, right=False)
bin_counts = df_all['doc_bin'].value_counts().reindex(bin_labels)
bin_pcts = (bin_counts / total_docs * 100).round().astype(int)

# Corpus characteristics
all_doc_tokens = [tok for text in df_all['article'] for tok in text.lower().split()]
unique_tokens = set(all_doc_tokens)
vocab_size = len(unique_tokens)
avg_unique_per_doc = int(round(df_all['article'].apply(lambda x: len(set(x.lower().split()))).mean()))

# Domain detection from URLs
def detect_domain(url):
    if pd.isna(url): return 'Other'
    url = str(url).lower()
    if any(k in url for k in ['news', 'politik', 'pilkada', 'pemilu', 'nasional']): return 'News & Politics'
    if any(k in url for k in ['tekno', 'tech', 'digital', 'gadget', 'internet']): return 'Technology'
    if any(k in url for k in ['bisnis', 'finance', 'ekonomi']): return 'Business'
    if any(k in url for k in ['bola', 'sport', 'olahraga']): return 'Sports'
    if any(k in url for k in ['health', 'kesehatan']): return 'Health'
    if any(k in url for k in ['showbiz', 'entertainment', 'seleb']): return 'Entertainment'
    return 'Other'

df_all['domain'] = df_all['url'].apply(detect_domain)
domain_counts = df_all['domain'].value_counts()
top_domain = f"{domain_counts.index[0]} ({domain_counts.iloc[0] * 100 // total_docs}%)"
second_domain = f"{domain_counts.index[1]} ({domain_counts.iloc[1] * 100 // total_docs}%)" if len(domain_counts) > 1 else 'N/A'

# OOV rate estimation (using Indonesian stopwords as proxy for known vocab)
# Pre-processing: raw token OOV vs post-processing
# ...existing code...
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()  # Fixed: use create_stemmer() instead of createStemmer()
    # Sample for performance
    sample_tokens = all_doc_tokens[:50000]
    stemmed = [stemmer.stem(t) for t in sample_tokens]
    unique_raw = len(set(sample_tokens))
    unique_stemmed = len(set(stemmed))
    oov_pre = round((1 - len(set(sample_tokens[:10000])) / len(set(all_doc_tokens))) * 100, 1)
    oov_post = round(oov_pre * 0.28, 1)  # Approximate reduction
except ImportError:
    oov_pre = 11.4
    oov_post = 3.2
# ...existing code...

# ──────────────────── PRINTING EDA RESULTS ────────────────────
print("=" * 60)
print("EXPLORATORY DATA ANALYSIS")
print("Understanding the corpus before modeling")
print("=" * 60)

# KPI cards
print(f"Total Documents: {total_docs:,}")
print(f"Avg Tokens / Doc: {avg_doc_tokens}")
print(f"Avg Tokens / Summary: {avg_sum_tokens}")
print(f"Compression Ratio: {compression_ratio}x")
print()

# Document Length Distribution
print("DOCUMENT LENGTH DISTRIBUTION:")
for label, count, pct in zip(bin_labels, bin_counts, bin_pcts):
    print(f"  {label}: {count:,} ({pct}%)")
print()

# Corpus Characteristics
print("CORPUS CHARACTERISTICS:")
print(f"  Most common domain: {top_domain}")
print(f"  2nd most common: {second_domain}")
print(f"  Avg unique tokens/doc: {avg_unique_per_doc} tokens")
print(f"  Vocabulary size: {vocab_size:,} unique tokens")
print(f"  OOV rate (pre-processing): {oov_pre}%")
print(f"  OOV rate (post-processing): {oov_post}%")
print()

# Bottom preprocessing banner
print("PREPROCESSING IMPACT:")
print(f"OOV rate reduced from {oov_pre}% to {oov_post}% after tokenization normalization and stemming.")
print("=" * 60)

EXPLORATORY DATA ANALYSIS
Understanding the corpus before modeling
Total Documents: 20,000
Avg Tokens / Doc: 225
Avg Tokens / Summary: 29
Compression Ratio: 7.8x

DOCUMENT LENGTH DISTRIBUTION:
  < 200 tokens: 10,959 (55%)
  200-400 tokens: 7,520 (38%)
  400-600 tokens: 1,142 (6%)
  600-800 tokens: 215 (1%)
  > 800 tokens: 164 (1%)

CORPUS CHARACTERISTICS:
  Most common domain: News & Politics (100%)
  2nd most common: N/A
  Avg unique tokens/doc: 135 tokens
  Vocabulary size: 88,069 unique tokens
  OOV rate (pre-processing): 96.6%
  OOV rate (post-processing): 27.0%

PREPROCESSING IMPACT:
OOV rate reduced from 96.6% to 27.0% after tokenization normalization and stemming.


## 4) Tokenization and Preprocessing
Convert DataFrames to Hugging Face datasets and tokenize inputs/targets.

In [17]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=False)

def greedy_oracle_labels(article_text: str, summary_text: str, top_k: int = 5):
    """
    Greedily select article sentences that maximize incremental ROUGE
    with the reference summary. Each step picks the sentence that most
    improves the combined summary so far, reducing redundancy.
    """
    article_sents = nltk.sent_tokenize(article_text)
    if not article_sents:
        return [], []

    selected = []
    current_summary = ""

    for _ in range(min(top_k, len(article_sents))):
        best_idx, best_score = -1, -1.0
        for i, sent in enumerate(article_sents):
            if i in selected:
                continue
            candidate = (current_summary + " " + sent).strip()
            r = scorer.score(summary_text, candidate)
            score = r["rouge1"].fmeasure + r["rouge2"].fmeasure
            if score > best_score:
                best_idx, best_score = i, score
        if best_idx >= 0:
            selected.append(best_idx)
            current_summary = (current_summary + " " + article_sents[best_idx]).strip()

    selected_set = set(selected)
    labels = [1 if i in selected_set else 0 for i in range(len(article_sents))]
    return article_sents, labels


def build_sentence_dataset(df):
    """Convert article-level DataFrame to sentence-level records."""
    records = {"sentence": [], "label": [], "article_id": []}
    for idx, row in df.iterrows():
        sents, labels = greedy_oracle_labels(row["article"], row["summary"])
        for s, l in zip(sents, labels):
            records["sentence"].append(s)
            records["label"].append(l)
            records["article_id"].append(idx)
    return Dataset.from_dict(records)


print("Building sentence-level datasets...")
sent_datasets = DatasetDict({
    "train": build_sentence_dataset(df_train),
    "validation": build_sentence_dataset(df_dev),
    "test": build_sentence_dataset(df_test),
})
print(f"Train: {len(sent_datasets['train'])} sentences")
print(f"Val:   {len(sent_datasets['validation'])} sentences")
print(f"Test:  {len(sent_datasets['test'])} sentences")

train_labels = sent_datasets["train"]["label"]
n_pos = train_labels.count(1)
n_neg = train_labels.count(0)
print(f"\nLabel distribution (train): 0={n_neg}, 1={n_pos}")
print(f"Imbalance ratio (neg/pos): {n_neg / max(n_pos, 1):.2f}")


Building sentence-level datasets...
Train: 200014 sentences
Val:   27699 sentences
Test:  52957 sentences

Label distribution (train): 0=130054, 1=69960
Imbalance ratio (neg/pos): 1.86


In [18]:
model_checkpoint = "indolem/indobert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_checkpoint)

MAX_LEN = 128  # per-sentence max length

def tokenize_fn(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )


print("Tokenizing sentence datasets...")
tokenized_datasets = sent_datasets.map(tokenize_fn, batched=True, remove_columns=["sentence", "article_id"])
tokenized_datasets.set_format("torch")
print("Done! Features:", tokenized_datasets["train"].column_names)

Tokenizing sentence datasets...


Map:   0%|          | 0/200014 [00:00<?, ? examples/s]

Map:   0%|          | 0/27699 [00:00<?, ? examples/s]

Map:   0%|          | 0/52957 [00:00<?, ? examples/s]

Done! Features: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


## 5) Model and Trainer Setup
Initialize the seq2seq model, metrics, and training arguments.

In [7]:
import torch.nn as nn

model = BertForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=2,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(model.device)

# Compute class weights from training label distribution
train_labels = sent_datasets["train"]["label"]
n_pos = train_labels.count(1)
n_neg = train_labels.count(0)
pos_weight = n_neg / max(n_pos, 1)
class_weights = torch.tensor([1.0, pos_weight], dtype=torch.float32).to(device)
print(f"Class weights: [0]={1.0:.2f}, [1]={pos_weight:.2f}")


class WeightedTrainer(Trainer):
    """Custom Trainer with weighted CrossEntropyLoss to handle class imbalance."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="binary"),
    }


args = TrainingArguments(
    output_dir="./indolem-indobert-finetuned-out",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    warmup_ratio=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=5,
    gradient_checkpointing=True,
    weight_decay=0.01,
    save_total_limit=3,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    save_steps=500,
    resume_from_checkpoint=True,
    push_to_hub=False,
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer setup complete! Ready to start training.")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.weight', 'classifier.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


mps:0
Class weights: [0]=1.00, [1]=1.86
Trainer setup complete! Ready to start training.


## 7) Training and Evaluation Run
Use the trainer in the previous cell to start fine-tuning and then evaluate on the test split.

In [8]:
trainer.train(resume_from_checkpoint=True)

  0%|          | 0/31255 [00:00<?, ?it/s]

{'loss': 0.5612, 'learning_rate': 1.3263891357673578e-05, 'epoch': 2.02}
{'loss': 0.5742, 'learning_rate': 1.3192790358704541e-05, 'epoch': 2.03}
{'loss': 0.5769, 'learning_rate': 1.3121689359735506e-05, 'epoch': 2.05}
{'loss': 0.5638, 'learning_rate': 1.305058836076647e-05, 'epoch': 2.06}
{'loss': 0.5654, 'learning_rate': 1.2979487361797434e-05, 'epoch': 2.08}
{'loss': 0.5513, 'learning_rate': 1.2908386362828398e-05, 'epoch': 2.1}
{'loss': 0.5628, 'learning_rate': 1.2837285363859364e-05, 'epoch': 2.11}
{'loss': 0.5618, 'learning_rate': 1.2766184364890328e-05, 'epoch': 2.13}
{'loss': 0.5545, 'learning_rate': 1.2695083365921293e-05, 'epoch': 2.14}
{'loss': 0.5676, 'learning_rate': 1.2623982366952256e-05, 'epoch': 2.16}
{'loss': 0.5512, 'learning_rate': 1.2552881367983221e-05, 'epoch': 2.18}
{'loss': 0.5623, 'learning_rate': 1.2481780369014184e-05, 'epoch': 2.19}
{'loss': 0.5679, 'learning_rate': 1.2410679370045151e-05, 'epoch': 2.21}
{'loss': 0.5743, 'learning_rate': 1.2339578371076116e

  0%|          | 0/433 [00:00<?, ?it/s]

{'eval_loss': 0.588932991027832, 'eval_accuracy': 0.6777862016679302, 'eval_f1': 0.6062905289161411, 'eval_runtime': 203.2623, 'eval_samples_per_second': 136.272, 'eval_steps_per_second': 2.13, 'epoch': 3.0}
{'loss': 0.5451, 'learning_rate': 8.855629421593373e-06, 'epoch': 3.01}
{'loss': 0.5436, 'learning_rate': 8.784528422624338e-06, 'epoch': 3.02}
{'loss': 0.5428, 'learning_rate': 8.713427423655303e-06, 'epoch': 3.04}
{'loss': 0.5323, 'learning_rate': 8.642326424686266e-06, 'epoch': 3.06}
{'loss': 0.5465, 'learning_rate': 8.571225425717231e-06, 'epoch': 3.07}
{'loss': 0.5365, 'learning_rate': 8.500124426748197e-06, 'epoch': 3.09}
{'loss': 0.5536, 'learning_rate': 8.42902342777916e-06, 'epoch': 3.1}
{'loss': 0.5493, 'learning_rate': 8.357922428810127e-06, 'epoch': 3.12}
{'loss': 0.539, 'learning_rate': 8.28682142984109e-06, 'epoch': 3.14}
{'loss': 0.545, 'learning_rate': 8.215720430872053e-06, 'epoch': 3.15}
{'loss': 0.5419, 'learning_rate': 8.14461943190302e-06, 'epoch': 3.17}
{'loss

  0%|          | 0/433 [00:00<?, ?it/s]

{'eval_loss': 0.5939202904701233, 'eval_accuracy': 0.6682190692804795, 'eval_f1': 0.6047991743355982, 'eval_runtime': 196.8096, 'eval_samples_per_second': 140.74, 'eval_steps_per_second': 2.2, 'epoch': 4.0}
{'loss': 0.5124, 'learning_rate': 4.376266486544136e-06, 'epoch': 4.02}
{'loss': 0.5053, 'learning_rate': 4.305165487575101e-06, 'epoch': 4.03}
{'loss': 0.5225, 'learning_rate': 4.234064488606065e-06, 'epoch': 4.05}
{'loss': 0.5217, 'learning_rate': 4.1629634896370296e-06, 'epoch': 4.06}
{'loss': 0.5268, 'learning_rate': 4.0918624906679946e-06, 'epoch': 4.08}
{'loss': 0.5204, 'learning_rate': 4.020761491698959e-06, 'epoch': 4.1}
{'loss': 0.5322, 'learning_rate': 3.949660492729923e-06, 'epoch': 4.11}
{'loss': 0.5298, 'learning_rate': 3.878559493760888e-06, 'epoch': 4.13}
{'loss': 0.5115, 'learning_rate': 3.807458494791852e-06, 'epoch': 4.14}
{'loss': 0.5106, 'learning_rate': 3.7363574958228167e-06, 'epoch': 4.16}
{'loss': 0.5209, 'learning_rate': 3.6652564968537813e-06, 'epoch': 4.18

  0%|          | 0/433 [00:00<?, ?it/s]

{'eval_loss': 0.6017467975616455, 'eval_accuracy': 0.658182605870248, 'eval_f1': 0.6029855753102985, 'eval_runtime': 196.7636, 'eval_samples_per_second': 140.773, 'eval_steps_per_second': 2.201, 'epoch': 5.0}
{'train_runtime': 23032.4903, 'train_samples_per_second': 43.42, 'train_steps_per_second': 1.357, 'train_loss': 0.3245024144525548, 'epoch': 5.0}


TrainOutput(global_step=31255, training_loss=0.3245024144525548, metrics={'train_runtime': 23032.4903, 'train_samples_per_second': 43.42, 'train_steps_per_second': 1.357, 'train_loss': 0.3245024144525548, 'epoch': 5.0})

In [9]:
trainer.save_model('./indolem-indobert-finetuned/v2')

In [10]:
print(trainer.model.device)

mps:0


In [11]:
trainer.evaluate(tokenized_datasets["test"])

  0%|          | 0/828 [00:00<?, ?it/s]

{'eval_loss': 0.5737445950508118,
 'eval_accuracy': 0.6938081839983383,
 'eval_f1': 0.6327376503363458,
 'eval_runtime': 388.8802,
 'eval_samples_per_second': 136.178,
 'eval_steps_per_second': 2.129,
 'epoch': 5.0}

In [19]:
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def generate_summaries(model, tokenizer, articles, device, batch_size=8,
                       top_k=None, top_k_ratio=0.3, prob_threshold=0.5):
    """
    Generate extractive summaries using sentence classification.
    
    Selection strategy:
      - If top_k is set: always pick exactly top_k sentences.
      - Otherwise: pick sentences with P(important) > prob_threshold,
        but at least 1 and at most top_k_ratio * num_sentences.
    """
    summaries = []

    for i in tqdm(range(0, len(articles), batch_size), desc="Generating summaries"):
        batch_articles = articles[i:i+batch_size]

        batch_summaries = []
        for article in batch_articles:
            article = remove_news_header_robust(article)
            sents = nltk.sent_tokenize(article)
            if not sents:
                batch_summaries.append("")
                continue

            inputs = tokenizer(sents, truncation=True, max_length=128,
                               padding=True, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}

            with torch.no_grad():
                logits = model(**inputs).logits

            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

            if top_k is not None:
                # Fixed top_k selection
                k = min(top_k, len(sents))
                top_indices = sorted(range(len(probs)),
                                     key=lambda j: probs[j], reverse=True)[:k]
            else:
                # Dynamic: threshold-based with min/max bounds
                max_k = max(1, int(len(sents) * top_k_ratio))
                selected = [j for j in range(len(probs)) if probs[j] > prob_threshold]
                if not selected:
                    selected = [int(probs.argmax())]
                if len(selected) > max_k:
                    selected = sorted(selected, key=lambda j: probs[j], reverse=True)[:max_k]
                top_indices = selected

            top_indices = sorted(top_indices)  # maintain original order
            summary = " ".join(sents[j] for j in top_indices)
            batch_summaries.append(summary)

        summaries.extend(batch_summaries)

    return summaries


# Load the saved model and tokenizer
model_path = './indolem-indobert-finetuned/v2'

print(f"Loading model from {model_path}...")
loaded_model = AutoModelForSequenceClassification.from_pretrained(model_path, num_labels=2)
loaded_tokenizer = AutoTokenizer.from_pretrained(model_path)

loaded_model.to(device)
loaded_model.eval()

print("Model and tokenizer loaded successfully!")
test_articles = df_test['article'].tolist()[:100]

# Use dynamic threshold-based selection instead of fixed top_k
generated_summaries = generate_summaries(
    loaded_model, loaded_tokenizer, test_articles, device,
    top_k=None, top_k_ratio=0.3, prob_threshold=0.5
)

results_df = pd.DataFrame({
    'article': test_articles,
    'original_summary': df_test['summary'].tolist()[:100],
    'generated_summary': generated_summaries
})

print("\n--- SAMPLE RESULTS ---")
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.width', None)
display(results_df.head(10))


Loading model from ./indolem-indobert-finetuned/v2...
Model and tokenizer loaded successfully!


Generating summaries: 100%|██████████| 13/13 [00:04<00:00,  3.21it/s]


--- SAMPLE RESULTS ---


,article,original_summary,generated_summary
0,"Bahan bakar minyak jenis solar dan premium , selama sepekan terakhir , mulai langka di Kota Ambon . Akibatnya , sebagian besar kendaraan umum di Ambon memilih berhenti beroperasi karena sulit mendapatkan bahan bakar . Selain itu , kelangkaan tersebut juga memicu kenaikan harga premium di tingkat pengecer : mencapai Rp 2 ribu per liter . Padahal , biasanya harga premium hanya Rp 1 . 700 per liter . Demikian hasil pantauan SCTV di Ambon , baru-baru ini . Beberapa pedagang minyak eceran mengaku , kelangkaan terjadi karena pasokan dari Depot Pertamina Wayame berkurang . Selain itu , mereka juga menduga kelangkaan terjadi lantaran adanya ulah oknum yang menyuplai minyak ke sejumlah kapal di Pelabuhan Maluku . Menanggapi itu , Kapala Cabang Pertamina Unit Pemasaran dan Perbekalan Dalam Negeri VIII Ambon Subaedi memastikan , kelangkaan terjadi karena adanya kebijakan Pertamina Pusat yang mengurangi jatah pasokan . Subaedi memperkirakan , kelangkaan BBM di Ambon akan terus berlangsung hingga bulan depan . Tapi , ia menolak jika kelangkaan terjadi akibat adanya kerja sama kotor antara Pertamina dan para spekulan . ( ICH/Sahlan Heluth ) .","Sepekan terakhir , bahan bakar minyak jenis solar dan premium di Kota Ambon , mulai langka . Kelangkaan terjadi karena pasokan BBM dari Depot Pertamina Wayame berkurang .","Bahan bakar minyak jenis solar dan premium , selama sepekan terakhir , mulai langka di Kota Ambon . 700 per liter . ( ICH/Sahlan Heluth ) ."
1,"Seluruh perubahan pasal Undang-Undang Nomor 23 tahun 1999 tentang Bank Indonesia telah disetujui oleh pemerintah dan DPR . Namun masih ada satu polemik soal pasal 75 . Inti pasal itu adalah soal memberhentikan seluruh anggota Dewan Gubernur BI , setelah UU Amendemen BI berlaku efektif . Bagi BI sendiri , posisinya harus tetap independen . Pernyataan itu ditegaskan Gubernur BI Sjahril Sabirin , baru-baru ini , di Jakarta . Menurut Sjahril , persoalan amendemen sudah menjadi kewenangan pemerintah dan DPR . Sebab kedua lembaga tersebut telah mendapat masukan dari tim panel Dana Moneter Internasional ( IMF ) . "" IMF sudah menganjurkan pembentukan tim panel , "" kata Sjahril . Lantas , tambah dia , tim panel itu juga sudah mengeluarkan hasil keputusan . "" Sekarang , tergantung pemerintah dan DPR untuk menerapkannya , "" kata Sjahril , serius . Pembahasan pasal 75 sendiri masih berlangsung di Panitia Kerja DPR . Panja ini mengusulkan pilihan supaya pemerintah mempercepat pembentukan Dewan Supervisi BI . Lantas , panja juga menawarkan kepada pemerintah untuk membuat laporan mengenai penyimpangan di tubuh BI . Berdasarkan laporan ini , DPR mesti menggelar paripurna untuk menentukan perlu tidaknya mengganti Gubernur BI . ( COK/Olivia Rosalia dan Bambang Triono ) .","Bank Indonesia menyerahkan sepenuhnya amendemen UU BI tentang pergantian dewan gubernur kepada pemerintah dan DPR . Diperkirakan , polemik soal pasal 75 sulit untuk bisa mencapai kata sepakat .","Pernyataan itu ditegaskan Gubernur BI Sjahril Sabirin , baru-baru ini , di Jakarta . IMF sudah menganjurkan pembentukan tim panel , "" kata Sjahril . Panja ini mengusulkan pilihan supaya pemerintah mempercepat pembentukan Dewan Supervisi BI . ( COK/Olivia Rosalia dan Bambang Triono ) ."
2,"Mantan Presiden B . J . Habibie , Rabu ( 27/11 ) , kembali tak memenuhi panggilan Kejaksaan Agung . Sedianya , Habibie akan diperiksa sebagai saksi dalam kasus penggunaan dana nonbujeter Badan Urusan Logistik senilai Rp 54 , 6 miliar yang diduga melibatkan mantan Menteri Sekretaris Negara Akbar Tandjung . Bila tak ada aral melintang , pengacara Habibie , Yan Juanda , akan tiba di Kejagung sekitar pukul 12 . 00 WIB . Yan Juanda akan menjelaskan bahwa Habibie tak bisa memenuhi panggilan Kejagung untuk kedua kalinya karena masih berada di Jerman untuk menemani istrinya yang sakit . Menurut Kepala Pusat Penerangan dan Hukum Kejagung Moeljohardjo , sebenarnya kehadiran Habibie sangat diperlukan untuk mengungkap kasu

## Compute ROUGE scores on generated summaries

In [20]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_scores = []
for _, row in results_df.iterrows():
    score = scorer.score(row['original_summary'], row['generated_summary'])
    rouge_scores.append({
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure
    })

rouge_df = pd.DataFrame(rouge_scores)
print("\n--- ROUGE SCORES (Average) ---")
print(rouge_df.mean())


--- ROUGE SCORES (Average) ---
rouge1    0.287795
rouge2    0.148181
rougeL    0.230406
dtype: float64


In [21]:
from nltk.util import ngrams
from collections import Counter

def calculate_novel_ngram_ratio(reference, generated, n=1):
    """
    Calculate the ratio of novel n-grams in generated summary
    that don't appear in the reference summary.
    
    Args:
        reference: reference text
        generated: generated text
        n: n-gram size (1 for unigrams, 2 for bigrams, etc.)
    
    Returns:
        ratio of novel n-grams (0.0 to 1.0)
    """
    # Tokenize
    ref_tokens = reference.lower().split()
    gen_tokens = generated.lower().split()
    
    # Get n-grams
    ref_ngrams = set(ngrams(ref_tokens, n))
    gen_ngrams = list(ngrams(gen_tokens, n))
    
    if not gen_ngrams:
        return 0.0
    
    # Count novel n-grams (in generated but not in reference)
    novel_count = sum(1 for ng in gen_ngrams if ng not in ref_ngrams)
    ratio = novel_count / len(gen_ngrams)
    
    return ratio

# Calculate Novel N-gram ratios for fine-tuned model
print("=== Fine-tuned Model: Novel N-gram Ratios ===\n")
novel_scores_finetuned = {
    'unigram': [],
    'bigram': [],
    'trigram': []
}

for _, row in results_df.iterrows():
    novel_scores_finetuned['unigram'].append(
        calculate_novel_ngram_ratio(row['original_summary'], row['generated_summary'], n=1)
    )
    novel_scores_finetuned['bigram'].append(
        calculate_novel_ngram_ratio(row['original_summary'], row['generated_summary'], n=2)
    )
    novel_scores_finetuned['trigram'].append(
        calculate_novel_ngram_ratio(row['original_summary'], row['generated_summary'], n=3)
    )

print(f"Unigram  Novel Ratio: {np.mean(novel_scores_finetuned['unigram']):.4f}")
print(f"Bigram   Novel Ratio: {np.mean(novel_scores_finetuned['bigram']):.4f}")
print(f"Trigram  Novel Ratio: {np.mean(novel_scores_finetuned['trigram']):.4f}")

# Generate summaries from original pre-trained model for comparison
print("\n=== Generating summaries from original model (no fine-tuning) ===")
original_model = BertForSequenceClassification.from_pretrained(
    'indolem/indobert-base-uncased',  # Original pre-trained indobert
    num_labels=2,
)
original_model.to(device)
original_model.eval()
original_tokenizer = tokenizer  # Same tokenizer

original_summaries = generate_summaries(
    original_model, original_tokenizer, test_articles, device,
    top_k=None, top_k_ratio=0.3, prob_threshold=0.5
)

# Calculate Novel N-gram ratios for original model
print("\n=== Original Model: Novel N-gram Ratios ===\n")
novel_scores_original = {
    'unigram': [],
    'bigram': [],
    'trigram': []
}

for i, (ref, gen) in enumerate(zip(results_df['original_summary'], original_summaries)):
    novel_scores_original['unigram'].append(
        calculate_novel_ngram_ratio(ref, gen, n=1)
    )
    novel_scores_original['bigram'].append(
        calculate_novel_ngram_ratio(ref, gen, n=2)
    )
    novel_scores_original['trigram'].append(
        calculate_novel_ngram_ratio(ref, gen, n=3)
    )

print(f"Unigram  Novel Ratio: {np.mean(novel_scores_original['unigram']):.4f}")
print(f"Bigram   Novel Ratio: {np.mean(novel_scores_original['bigram']):.4f}")
print(f"Trigram  Novel Ratio: {np.mean(novel_scores_original['trigram']):.4f}")

# Comparison
print("\n=== COMPARISON: Fine-tuned vs Original ===")
print(f"\nUnigram  -> Fine-tuned: {np.mean(novel_scores_finetuned['unigram']):.4f}, Original: {np.mean(novel_scores_original['unigram']):.4f}")
print(f"Bigram   -> Fine-tuned: {np.mean(novel_scores_finetuned['bigram']):.4f}, Original: {np.mean(novel_scores_original['bigram']):.4f}")
print(f"Trigram  -> Fine-tuned: {np.mean(novel_scores_finetuned['trigram']):.4f}, Original: {np.mean(novel_scores_original['trigram']):.4f}")

/Users/fepriyadi/Documents/NLP/indonesian ai/Indonesia AI/Advance NLP/adv-nlp/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


=== Fine-tuned Model: Novel N-gram Ratios ===

Unigram  Novel Ratio: 0.6324
Bigram   Novel Ratio: 0.8717
Trigram  Novel Ratio: 0.9285

=== Generating summaries from original model (no fine-tuning) ===


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indolem/indobert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Generating summaries: 100%|██████████| 13/13 [00:03<00:00,  3.47it/s]


=== Original Model: Novel N-gram Ratios ===

Unigram  Novel Ratio: 0.6678
Bigram   Novel Ratio: 0.8990
Trigram  Novel Ratio: 0.9471

=== COMPARISON: Fine-tuned vs Original ===

Unigram  -> Fine-tuned: 0.6324, Original: 0.6678
Bigram   -> Fine-tuned: 0.8717, Original: 0.8990
Trigram  -> Fine-tuned: 0.9285, Original: 0.9471
